# F1 · PCA or feature selection? Three data shapes, five strategies

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Compare five dimensionality reduction strategies in front of a logistic regression (no reduction, `PCA(5)`, `PCA(15)`, selection of 5 features with `SelectKBest`, `PCA(5)` without a scaler) on three kinds of data: **redundant** features (5 latent factors in 40 columns), signal in
**low-variance directions** and **sparse** signal (5 informative columns out of 60). Average over 3 datasets per kind.

## Dataset

`generate(kind, seed)` (given): 400 examples per dataset; `kind` in `"redundant"`, `"low variance"`, `"sparse"`. Final model: `LogisticRegression(max_iter=2000)`; validation `StratifiedKFold(5, shuffle=True, random_state=0)`, accuracy.

## Educational purpose

Replace "use PCA to reduce the dimensions" with a rule based on the shape of the data: PCA helps (or at least does no harm) when variance and information coincide; it hurts when the signal is in a few columns among many or in low-variance directions. Supervised feature
selection is often a better fit.

## Task

1. Build `models` (dictionary name → function that creates the `Pipeline`): `none` (`StandardScaler` → logistic), `PCA(5)` and `PCA(15)` (`StandardScaler` → `PCA(k)` → logistic), `Select(5)` (`StandardScaler` → `SelectKBest(f_classif, k=5)` → logistic) and `PCA(5) raw` (`PCA(5)` → logistic, no scaler).
2. Build `table` (index: the three kinds of data; columns: the five models) with the cross-validation accuracy **averaged over `seed` = 0, 1, 2**.
3. Save `best` (dictionary kind → best model), `pca_loses` (sorted list of the kinds where `PCA(5)` is worse than `none` by more than 0.05) and `pca_ties` (the kinds where the difference between `PCA(5)` and `none` is less than 0.03).
4. Save `scaler_matters` (dictionary kind → `PCA(5)` with scaler minus `PCA(5) raw`): where does scaling the PCA change the result the most?

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def generate(kind, seed=0, n=400):
    rng = np.random.default_rng(seed)
    if kind == "redundant":                        # 5 latent factors explain 40 correlated columns; y depends on 2 factors
        L = rng.normal(size=(n, 5))
        X = L @ rng.normal(size=(5, 40)) + rng.normal(0, 1, (n, 40))
        y = (L[:, 0] + L[:, 1] + rng.normal(0, 1, n) > 0).astype(int)
    elif kind == "low variance":                   # 20 high-variance noise columns and 3 low-variance informative ones
        z = rng.normal(size=(n, 3))
        y = (z.sum(1) + rng.normal(0, 0.5, n) > 0).astype(int)
        X = np.c_[rng.normal(0, 5, (n, 20)), 0.5 * z]
    else:                                          # 5 informative columns out of 60
        X = rng.normal(size=(n, 60))
        y = (X[:, :5].sum(1) + rng.normal(0, 1, n) > 0).astype(int)
    return X, y


cv = StratifiedKFold(5, shuffle=True, random_state=0)
kinds = ["redundant", "low variance", "sparse"]

## Your work

In [ ]:
models = {
    "none": lambda: ...,
}

In [ ]:
table = pd.DataFrame(index=kinds, columns=list(models), dtype=float)
for kind in kinds:
    for name, create in models.items():
        ...
table.round(3)

In [ ]:
best = {}
pca_loses = None
pca_ties = None
scaler_matters = {}

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Example: `"PCA(5)": lambda: make_pipeline(StandardScaler(), PCA(5), LogisticRegression(max_iter=2000))`. Average over the seeds: `np.mean([cross_val_score(create(), *generate(kind, seed), cv=cv).mean() for seed in range(3)])`. Kinds where PCA(5) loses: `sorted(t for t in kinds if table.loc[t, "PCA(5)"] < table.loc[t, "none"] - 0.05)`.'):
    check(list(models) == ["none", "PCA(5)", "PCA(15)", "Select(5)", "PCA(5) raw"], "five models with the given names", "none, PCA(5), PCA(15), Select(5), PCA(5) raw")
    check(table.shape == (3, 5) and table.notna().all().all(), "complete 3 x 5 table", "one accuracy for each kind and model")
    check(table.loc["low variance", "PCA(5)"] < table.loc["low variance", "none"] - 0.1 and table.loc["sparse", "PCA(5)"] < 0.6, "PCA(5) fails with low-variance signal (over 0.1 below 'none') and with sparse signal (below 0.6, almost chance)", "compare the rows of the table")
    check(pca_loses == ["low variance", "sparse"] and pca_ties == ["redundant"], "PCA(5) loses on 'low variance' and 'sparse'; it ties on 'redundant'", "difference between PCA(5) and none")
    check(best["low variance"] == "Select(5)" and best["sparse"] == "Select(5)", "supervised selection is the best when the signal is in a few columns", "idxmax per row")
    check(scaler_matters["low variance"] > 0.1 and abs(scaler_matters["redundant"]) < 0.05, "the scaler matters a lot for 'low variance' (over 0.1) and almost nothing for 'redundant'", "PCA(5) with scaler minus PCA(5) raw")
done()

---
The commented solution is in `solutions/F_compare/F1_pca_vs_feature_selection.ipynb`. Open it after you pass the check, or after you have really tried.